# NB2c — Ablation deep-eval (Bolivia holdout + flood-only)

Re-evaluates the 18 trained ablation checkpoints on the **harder splits** to
decide whether FiLM and the curriculum earn their place:

- **Bolivia holdout** — out-of-region generalization (never seen in training)
- **flood-only IoU** — permanent water (JRC) excluded, so credit is only for actual flood

No retraining — loads each `best.pt`, rebuilds the arch from its stored config,
loads the EMA weights, and runs the three-regime eval. ~10 min.

**Attach:** `lgmf_c` (v4), `lgmf-index`, `lgmf-ckpts` (the saved ablation output).
Internet can be **Off** here (we skip the pretrained download — trained weights overwrite it).


In [ ]:
import sys, os, glob, json, dataclasses
import numpy as np, torch
from torch.utils.data import DataLoader
init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "attach lgmf_c"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))
import lgmf
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11
from lgmf.model import build_model
from lgmf.engine import evaluate
print("lgmf", lgmf.__version__)

# locate index (val/bolivia + stats) and the checkpoint tree
idx_dir = os.path.dirname(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)[0])
stats = json.load(open(f"{idx_dir}/norm_stats.json"))
ckpts = glob.glob("/kaggle/input/**/best.pt", recursive=True)
assert ckpts, "no best.pt found -- attach lgmf-ckpts"
print(f"index: {idx_dir}\ncheckpoints found: {len(ckpts)}")

## Build the val + Bolivia loaders once (shared across all models)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"

base = LGMFConfig()
base.s1_mean, base.s1_std = stats["s1_mean"], stats["s1_std"]
base.s2_mean, base.s2_std = stats["s2_mean"], stats["s2_std"]
base.num_workers = 0                      # avoid the rasterio fork deadlock

def loader(split):
    ds = Sen1Floods11(base, f"{idx_dir}/{split}_index.json", augment=False)
    return DataLoader(ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=pin), len(ds)

val_ld, n_val = loader("val")
bol_ld, n_bol = loader("bolivia")
print(f"val {n_val} chips | bolivia {n_bol} chips")

## Load each checkpoint, rebuild arch, evaluate on both splits

In [ ]:
FIELDS = {f.name for f in dataclasses.fields(LGMFConfig)}
def cfg_from_dict(d):
    c = LGMFConfig(**{k: v for k, v in d.items() if k in FIELDS})
    c.s2_bands = tuple(c.s2_bands)
    c.encoder_pretrained = False          # trained weights overwrite init -> skip HF download
    return c

def parse(path):                          # .../<config>/seed<seed>/best.pt
    seed_dir = os.path.dirname(path)
    return os.path.basename(os.path.dirname(seed_dir)), os.path.basename(seed_dir)

regs = ["both", "s1_only", "s2_only"]
records = {}   # config -> list of {seed, val:{...}, bol:{...}}
for path in sorted(ckpts):
    config, seed = parse(path)
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg_i = cfg_from_dict(ck["cfg"])
    model = build_model(cfg_i).to(device)
    model.load_state_dict(ck.get("ema") or ck["model"])   # EMA = what we reported
    val = evaluate(model, val_ld, base, device)
    bol = evaluate(model, bol_ld, base, device)
    def pack(res):
        return {r: {"iou": res[r]["iou"],
                    "flood": res[r].get("flood_only_iou")} for r in regs}
    records.setdefault(config, []).append({"seed": seed, "val": pack(val), "bol": pack(bol)})
    print(f"  {config:8s} {seed:7s} | val s1 {val['s1_only']['iou']:.3f} "
          f"| bol s1 {bol['s1_only']['iou']:.3f}")
print("done evaluating", sum(len(v) for v in records.values()), "checkpoints")

## Aggregate mean ± std and compare to the full model

In [ ]:
def stack(config, split, reg, key="iou"):
    return np.array([r[split][reg][key] for r in records[config]
                     if r[split][reg][key] is not None])

CONFIGS = ["full", "no_film", "concat", "sum", "uniform", "no_cons"]
CONFIGS = [c for c in CONFIGS if c in records]

def ms(config, split, reg, key="iou"):
    a = stack(config, split, reg, key)
    return (float(a.mean()), float(a.std())) if len(a) else (float("nan"), 0.0)

def mean_of_means(config, split):
    a = np.array([np.mean([r[split][reg]["iou"] for reg in regs]) for r in records[config]])
    return float(a.mean()), float(a.std())

def row(config, split):
    b = ms(config, split, "both"); s1 = ms(config, split, "s1_only"); s2 = ms(config, split, "s2_only")
    m = mean_of_means(config, split)
    return b, s1, s2, m

def cell(t): return f"{t[0]:.3f}±{t[1]:.3f}"

for split, title in [("val", "VALIDATION (recap)"), ("bol", "BOLIVIA HOLDOUT (out-of-region)")]:
    fm = mean_of_means("full", split)
    print("\n" + "=" * 96)
    print(f"{title}")
    print(f"{'config':10s} {'both':>13s} {'s1_only':>13s} {'s2_only':>13s} {'mean':>13s}   {'d(mean)':>9s}")
    print("-" * 96)
    for c in CONFIGS:
        b, s1, s2, m = row(c, split)
        if c == "full":
            d = "  (ref)"
        else:
            dm = m[0] - fm[0]
            sig = "*" if abs(dm) > max(fm[1], m[1]) else " "
            d = f"{dm:+.3f}{sig}"
        print(f"{c:10s} {cell(b):>13s} {cell(s1):>13s} {cell(s2):>13s} {cell(m):>13s}   {d:>9s}")

# focused SAR-only view: val IoU, val flood-only, bolivia IoU -- the decision metric
print("\n" + "=" * 96)
print("SAR-only (the regime FiLM / gate / curriculum target)")
print(f"{'config':10s} {'val s1':>13s} {'val s1 flood':>13s} {'bolivia s1':>13s}   {'d val-s1':>9s}")
print("-" * 96)
f_v = ms("full", "val", "s1_only"); 
for c in CONFIGS:
    v = ms(c, "val", "s1_only"); vf = ms(c, "val", "s1_only", "flood"); bv = ms(c, "bol", "s1_only")
    if c == "full":
        d = "  (ref)"
    else:
        dm = v[0] - f_v[0]; sig = "*" if abs(dm) > max(f_v[1], v[1]) else " "
        d = f"{dm:+.3f}{sig}"
    print(f"{c:10s} {cell(v):>13s} {cell(vf):>13s} {cell(bv):>13s}   {d:>9s}")
print("=" * 96)

json.dump({"records": records}, open("/kaggle/working/deep_eval.json", "w"), indent=2, default=str)
print("\nsaved deep_eval.json")
print("* = |d| exceeds the larger seed-std (indicative; n=3). Bolivia n=15 chips -> noisy.")

## How to read this (keep-or-drop decision)

- **Look at where each component's Δ is significant.** The renorm gate vs `sum`
  and consistency (`no_cons`) already showed the clearest SAR-only effects on val.
- **FiLM (`no_film`) and curriculum (`uniform`):** if they lower SAR-only /
  Bolivia here too, keep them and frame as "helps the hard regimes." If they're
  flat or better *everywhere* (val + bolivia + flood-only), the honest move is to
  **drop the weaker one** — a simpler model is a stronger lightweight paper.
- **Bolivia is only 15 chips**, so treat it as directional, not decisive; weight
  the val flood-only column more.
